# Declaration of Originality

**School of Informatics & IT**
<br/>**Diploma in Applied Artificial Intelligence**
<br/>**Machine Learning for Developers (CAI2C08)**
<br/>**AY2026/2027 April Semester**
<br/>**Program Codes**

* Student Name: Grishm Chandru Mirpuri



**Declaration of Originality**
* I am the originator of this work, and I have appropriately acknowledged all other original sources used as my references for this work.
* I understand that Plagiarism is the act of taking and using the whole or any part of another person’s work, including work generated by AI, and presenting it as my own.
* I understand that Plagiarism is an academic offence and if I am found to have committed or abetted the offence of plagiarism in relation to this submitted work, disciplinary action will be enforced.

# Libraries

In [ ]:
## Import libraries

# Data Handling
import pandas as pd
import numpy as np

# Data splitting
from sklearn.model_selection import train_test_split

# Scaling
from sklearn.preprocessing import StandardScaler

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Model evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

# Save final model
import joblib

# 1. Business Understanding
The goal of this project is to build a machine learning model that can predict whether a hotel booking is likely to be cancelled.
Hotel booking cancellations can affect hotel revenue and room planning. If hotels can identify bookings with higher cancellation risks earlier, staff can take suitable actions such as sending confirmation reminders, reviewing high-risk bookings or planning room availability more carefully.
This project is a binary classification model because the model predicts one of two outcomes:
- Canceled
- Not_Canceled

The target column is "booking status"
The final solution will be deployed as a Streamlit web application where users can enter booking details and recieve a cancelling prediction, cancellation probability, risk level and simple business recommendation.

# 2. Data Understanding

## 2.1 Load dataset

In [ ]:
## Read *.csv file into pandas DataFrame
df = pd.read_csv("../data/hotel_bookings.csv")

# Display the first 5 rows of the DataFrame
df.head()

The dataset was loaded using pandas. The first 5 rows were displayed to confirm that the file was imported correctly.

In [ ]:
# check the number of rows and columns in the DataFrame
df.shape

The dataset contains 119390 rows and 32 columns. This confirms that the dataset has more than 200 records and is suitable for the project requirement.

In [ ]:
# display all column names in the DataFrame
df.columns

The column names were checked to understand the available features and to identify the target column for prediction. The target column for this project is "is_canceled".

In [ ]:
# Display basic information about the DataFrame
df.info()

The dataset information was checked to understand the data types and non null values before cleaning.

## 2.2 Summary Statistics

In [ ]:
## Understand the type of variable for each column
# display summary statistics for numerical columns
df.describe().T

In [ ]:
df["customer_type"].value_counts()

The summary statistics were checked to understand the range, average and spread of the numerical columns. This helps identify possible numerical values before data cleaning and modelling.

In [ ]:
# display summary statistics for categorical columns
df.describe(include='object').T

The categorical summary was checked to understand to check the main categories of the dataset and whether any columns may need cleaning before encoding.

In [ ]:
## Check for missing data
# check for missing values in each column
df.isnull().sum()

Missing values were checked because machine learning models cannot handle missing values directly. Any missing values ("children", "country", "agent" and "company") found will need to be handled during data cleaning.

In [ ]:
# check for duplicate rows in the DataFrame
df.duplicated().sum()

The dataset shows 31994 duplicated rows. However, the dataset does not contain a unique booking ID, so rows with the same values may still represent separate hotel bookings with similar details.
Therefore, I will not remove duplicated rows at this stage.

## 2.3 Exploratory Data Analysis using Summary Tables

### 2.3.1 Understanding distribution of data

### 2.3.1.1 Understanding distribution of target

In [ ]:
## Understanding distribution of target
target_count = df["is_canceled"].value_counts()
target_percentage = df["is_canceled"].value_counts(normalize=True) * 100
print("Target variable distribution:")
print(target_count)
print("\nTarget variable distribution (percentage):")
print(target_percentage)


The target distribution shows the number and percentage of cancelled and not cancelled bookings. This helps to check whether the classification problem is balanced or imbalanced. Since cancellation prediction is the business focus, accuracy alone may not be enough. Precision, recall and f1-score will also be considered.

### 2.3.1.2 Understanding distribution of features

In [ ]:
# Check correlation between numerical features and target variable

corr_with_target = df.corr(numeric_only=True)["is_canceled"].sort_values(ascending=False)

corr_table = pd.DataFrame({
    "Correlation with is_canceled": corr_with_target.round(3)
})

corr_table

Correlation was used to check which numerical features have a stronger relationship with the target column `is_canceled`.

Features with correlation values further away from 0 were considered for further EDA because they may have a stronger relationship with cancellation. Based on the correlation results and business meaning, I selected `lead_time`, `total_of_special_requests` and `required_car_parking_spaces` for further EDA.

These features were chosen because they relate to booking timing, guest commitment and customer travel planning. Correlation does not prove cause and effect, so these features will be explored further using summary tables.

In [ ]:
# Create lead time groups to understand cancellation distribution by lead time
df["lead_time_group"] = pd.cut(df["lead_time"], bins=[-1, 30, 90, 180, 365, np.inf], labels=["0-30", "31-90", "91-180", "181-365", "366+"])
cancellation_by_lead_time = df.groupby("lead_time_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by lead time group:")
print(cancellation_by_lead_time)

This table shows how cancellation rate changes across different lead time groups. Lead time is important because bookings made far in advance seems to have higher cancellation rates compared to bookings made closer to the arrival date.

In [ ]:
# Cancellation rate by number of special requests
cancellation_by_special_requests = df.groupby("total_of_special_requests")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by number of special requests:")
print(cancellation_by_special_requests)

This table compares cancellation rates across different numbers of special requests. Guests who make more special requests may be more committed to their booking, so this feature may help predict cancellation risk.

In [ ]:
# Count bookings by required car parking spaces
parking_space_counts = df["required_car_parking_spaces"].value_counts()
print("\nBooking counts by required car parking spaces:")
print(parking_space_counts)

In [ ]:
# Cancelation rate by required car parking spaces
cancellation_by_parking_spaces = df.groupby("required_car_parking_spaces")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by required car parking spaces:")
print(cancellation_by_parking_spaces)

The count table shows that most bookings required either 0 or 1 car parking space. Very few bookings required 2, 3 or 8 parking spaces.

Although bookings with parking spaces appear to have lower cancellation rates, some categories have very small counts. This means the cancellation percentages for those categories may not be reliable.
Therefore, "required_car_parking_spaces" will not be selected as a main feature for detailed EDA and for modelling later.

## 2.3.4 EDA on Selected Categorical Features

In [ ]:
# Count bookings by hotel type
hotel_type_counts = df["hotel"].value_counts()
print("\nBooking counts by hotel type:")
print(hotel_type_counts)

In [ ]:
# Cancellation rate by hotel type
cancellation_by_hotel = df.groupby("hotel")["is_canceled"].value_counts(normalize=True).unstack()
print("Cancellation distribution by hotel type:")
print(cancellation_by_hotel)

The count table shows that the dataset contains bookings from both city hotels and resort hotels.
The cancelation rate differs between the two hotel types, suggesting that "hotel" may be useful as a categorical feature for modelling after encoding. This also makes business sense because city hotels and resort hotels may have different customer behavior and cancelation patterns.

In [ ]:
# Count bookings by deposit type
deposit_type_counts = df["deposit_type"].value_counts()
print("\nBooking counts by deposit type:")
print(deposit_type_counts)

The count of bookings by deposit type was checked first to understand how common each deposit category is.

In [ ]:
# Cancellation rate by deposit type
cancellation_by_deposit_type = df.groupby("deposit_type")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by deposit type:")
print(cancellation_by_deposit_type)

The count table shows that most bookings are under the "No Deposit" category, while the other deposit categories have fewer records. This means the cancellation percentages should be interpreted together with the number of bookings in each category.
The cancellation rate differs across deposit types. This suggests that "deposit_type" may be used as a categorical feature for modelling after encoding. This also makes business sense because deposit conditions may affect whether guests cancel their bookings.

In [ ]:
# Count bookings by market segment
market_segment_counts = df["market_segment"].value_counts()
print("\nBooking counts by market segment:")
print(market_segment_counts)

In [ ]:
# Cancellation rate by market segment
cancellation_by_market_segment = df.groupby("market_segment")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by market segment:")
print(cancellation_by_market_segment)

The count table shows how bookings are distributed across different market segments. This is important because categories with fewer records may give less reliable percentages.
The cancelation rate differs across market segments. This suggests that "market_segment" may be useful as a categorical feature for modelling after encoding. This also makes business sense because different booking sources or customer groups may have different cancelation behavior.

In [ ]:
# Count bookings by customer type
customer_type_counts = df["customer_type"].value_counts()
print("\nBooking counts by customer type:")
print(customer_type_counts)

In [ ]:
# Cancelation rate by customer type
cancellation_by_customer_type = df.groupby("customer_type")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by customer type:")
print(cancellation_by_customer_type)

The count table shows how bookings are distributed across different customer types. This helps check whether each customer type has enough fecords for the cancellation percentages to be meaningful.
The cancellation rate differs across customer types. This sugggsts that "customer_type" may be useful as a categorical feature for modelling after encoding. This also makes business sense because individual, group and contract-related bookings may have different cancellation behavior.

## 2.3.5 EDA using Feature-Engineered Columns

In [ ]:
# Create simple feature-engineering columns based on existing columns
df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
df["total_guests"] = df["adults"] + df["children"] + df["babies"]
df["has_children"] = np.where(df["children"] + df["babies"] > 0, 1, 0)
df["has_special_request"] = np.where(df["total_of_special_requests"] > 0, 1, 0)

The engineered features were created to make the booking information easier to understand and more useful for modelling.

`total_nights` combines weekend nights and week nights into one feature that represents the full length of stay.

`total_guests` combines adults, children and babies into one feature that represents the total number of guests in the booking.

`has_children` simplifies the children and babies columns into a yes/no feature. This helps check whether family-related bookings have different cancellation behaviour.

`has_special_request` simplifies the number of special requests into a yes/no feature. This helps check whether guests who make at least one special request have different cancellation behaviour.

In [ ]:
# Create total nights groups for EDA
df["total_nights_group"] = pd.cut(df["total_nights"], bins=[-1, 1, 3, 7, 14, np.inf], labels=["0-1", "2-3", "4-7", "8-14", "15+"])

# Count bookings by total nights group
booking_counts_by_total_nights_group = df.groupby("total_nights_group").size()
print("\nBooking counts by total nights group:")
print(booking_counts_by_total_nights_group)

# Cancellation rate by total nights group
cancellation_by_total_nights_group = df.groupby("total_nights_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by total nights group:")
print(cancellation_by_total_nights_group)

The cancellation rate differs across total night groups. Short stays of 0–1 nights have a lower cancellation rate, while longer stays, especially 15+ nights, show a higher cancellation rate. This suggests that `total_nights` may be useful as an engineered feature for modelling.

However, the 15+ nights group should be interpreted carefully because longer-stay bookings may have fewer records than shorter-stay bookings.

In [ ]:
# Create total guests groups for EDA
df["total_guests_group"] = pd.cut(df["total_guests"], bins=[-1, 1, 2, 4, 6, np.inf], labels=["0", "1", "2", "3-4", "5+"])

# Count bookings by total guest group
booking_counts_by_total_guests_group = df.groupby("total_guests_group").size()
print("\nBooking counts by total guests group:")
print(booking_counts_by_total_guests_group)

# Cancellation rate by total guests group
cancellation_by_total_guests_group = df.groupby("total_guests_group")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by total guests group:")
print(cancellation_by_total_guests_group)

The original `total_guests` values had some very small groups, especially for larger guest numbers. Therefore, the values were grouped to make the pattern easier to interpret.

The count table shows that most bookings are for 1 to 4 guests, while 5+ guest bookings are much less common. The cancellation rate differs across guest groups, suggesting that `total_guests` may be useful as an engineered feature for modelling.

Bookings with 0 guests may represent unusual or invalid records, so they will be reviewed later during data cleaning.

In [39]:
# Count bookings by whether the booking includes children or babies
children_flag_counts = df["has_children"].value_counts().sort_index()
print("Booking counts by has_children")
print(children_flag_counts)

# Cancellation rate by whether the booking includes children or babies
cancellation_by_children_flag = df.groupby("has_children")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by has_children")
print(cancellation_by_children_flag)

Booking counts by has_children
has_children
0    110058
1      9332
Name: count, dtype: int64

Cancellation distribution by has_children
is_canceled          0         1
has_children                    
0             0.627787  0.372213
1             0.650772  0.349228


The count table shows that most bookings do not include children or babies. The cancellation rate is slightly lower for bookings with children or babies, but the difference is small.

This suggests that `has_children` may not be a strong feature on its own. It can still be considered during modelling, but it should not be treated as one of the main predictors based on EDA alone.

In [50]:
# Count bookings by whether the booking has special requests
special_requests_flag_counts = df["has_special_requests"].value_counts().sort_index()
print("Booking counts by has_special_request")
print(special_requests_flag_counts)

# Cancellation rate by whether the booking has special requests
cancellation_by_special_requests_flag = df.groupby("has_special_requests")["is_canceled"].value_counts(normalize=True).unstack()
print("\nCancellation distribution by has_special_request")
print(cancellation_by_special_requests_flag)

Booking counts by has_special_request
has_special_requests
0    70318
1    49072
Name: count, dtype: int64

Cancellation distribution by has_special_request
is_canceled                  0         1
has_special_requests                    
0                     0.522796  0.477204
1                     0.782605  0.217395


The count table shows that there are many bookings both with and without special requests, so the comparison is meaningful.

The cancellation rate is much lower for bookings with at least one special request. Bookings without special requests have about 47.8% cancellations, while bookings with special requests have about 21.7% cancellations.

This suggests that `has_special_request` is a useful engineered feature for modelling. It may indicate stronger guest commitment because guests who make special requests may be more likely to follow through with their booking.

## 2.3.6 Summary of EDA Findings

The EDA was carried out using summary tables.

The target distribution showed that the dataset contains both cancelled and non-cancelled bookings, which are imbalanced. Since the project focuses on predicting cancellation, accuracy alone may not be enough. Precision, recall, F1-score and confusion matrix will also be used later during model evaluation.

For numerical features, correlation was used to identify features that may have a relationship with `is_canceled`. Based on the correlation results and business meaning, `lead_time` and `total_of_special_requests` were explored further. The EDA showed that cancellation rates differ across lead time groups and number of special requests, suggesting that these features may be useful for modelling.

For categorical features, `hotel`, `deposit_type`, `market_segment` and `customer_type` were explored using count tables and cancellation rate tables. The cancellation rates differed across these categories, suggesting that these features may be useful after encoding with `pd.get_dummies()`.

Feature-engineered columns were also created during EDA. `total_nights`, `total_guests`, `has_children` and `has_special_request` were explored to see whether simplified booking features could reveal useful cancellation patterns. `total_nights`, `total_guests` and `has_special_request` showed useful differences in cancellation behaviour. `has_children` showed only a small difference, so it will not be treated as a main predictor based on EDA alone.

Overall, the EDA suggests that both selected original features and selected engineered features may be useful for predicting hotel booking cancellation. The final usefulness of these features will be checked later during model training and comparison.

### 2.3.2 Understanding relationship between variables

In [ ]:
## Understanding relationship between variables


# 3. Data Preparation

## 3.1 Data Cleaning

Data cleaning was carried out to prepare the dataset for modelling. This includes handling missing values, reviewing duplicate rows, removing invalid records and dropping columns that may cause data leakage.

In [62]:
# Create a copy of the dataset for cleaning and preprocessing
df_model = df.copy()

A copy of the original dataset was created so that data preparation changes do not affect the original EDA dataset.

In [63]:
# Check missing values before cleaning
missing_values_before_cleaning = df_model.isnull().sum().sort_values(ascending=False).head(10)
print("Missing values before cleaning:")
print(missing_values_before_cleaning)

Missing values before cleaning:
company                      112593
agent                         16340
country                         488
children                          4
total_guests                      4
total_guests_group                4
hotel                             0
arrival_date_day_of_month         0
arrival_date_week_number          0
arrival_date_month                0
dtype: int64


The missing value check shows that `company`, `agent`, `country` and `children` contain missing values.

The columns `company`, `agent` and `country` are not part of the selected features for this model, so they will not be handled individually at this stage. They will be excluded later during feature selection.

In [64]:
# Fill missing values in children with 0 (assuming missing means no children)
df_model["children"].fillna(0, inplace=True)


C:\Users\grish\AppData\Local\Temp\ipykernel_2420\629378725.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_model["children"].fillna(0, inplace=True)


The `children` column is needed to create engineered features such as `total_guests` and `has_children`. Therefore, missing values in `children` will be filled with 0, as a missing value likely means no children were recorded for the booking.

In [65]:
# Create total_guest temporarily to identify bookings with 0 guests (which is invalid)
df_model["total_guests"] = df_model["adults"] + df_model["children"] + df_model["babies"]
# Count bookings with 0 guests
zero_guest_bookings = df_model[df_model["total_guests"] == 0].shape[0]
print(f"\nNumber of bookings with 0 guests: {zero_guest_bookings}")


Number of bookings with 0 guests: 180


In [66]:
# Remove bookings with 0 guests from the dataset
df_model = df_model[df_model["total_guests"] > 0]

In [67]:
# Check dataset size after removing bookings with 0 guests
print(f"\nDataset size after removing bookings with 0 guests: {df_model.shape}")


Dataset size after removing bookings with 0 guests: (119210, 39)


Rows with 0 total guests were removed because they are likely invalid or unusual records. Removing them helps keep the modelling dataset more realistic.

In [68]:
# Check duplicate rows in modelling dataset
duplicate_rows = df_model.duplicated().sum()
print(f"\nNumber of duplicate rows in modelling dataset: {duplicate_rows}")


Number of duplicate rows in modelling dataset: 31980


Duplicate rows were checked but not removed. The dataset does not contain a unique booking ID, so identical rows may still represent separate hotel bookings with the same characteristics. Removing all duplicates may remove valid booking records and change the data distribution.

In [69]:
# Drop columns that may cause data leakage
columns_to_drop = ["reservation_status", "reservation_status_date"]
df_model = df_model.drop(columns=columns_to_drop, axis=1, errors='ignore')  # Use errors='ignore' to avoid KeyError if columns are not present  

`reservation_status` and `reservation_status_date` were removed because they may reveal information about the final booking outcome.

## 3.2 Train-Test Split

In [ ]:
## Split data into train set and test set


# 4. Modelling

### 4.2 Train Model

In [ ]:
## Initialise and train model


# 5. Model Evaluation

In [ ]:
## Evaluate model


In [ ]:
## New data

## Predict


## Iterative model development


In [ ]:
## Further feature engineering / feature selection